In [ ]:
import pandas as pd
import numpy as np
import xlwings as xw

crm_file = r"C:\Users\Daniel\Desktop\code\pcl\CRM\LBF\NEW_EXCEL\LBF_CRM_28_11_2025.xlsx"
addin_path = r"C:\Users\Daniel\AppData\Roaming\Microsoft\AddIns\BranchFunctions.xlam"

# Step 1 — Excel recalculation
app = xw.App(visible=False)
app.display_alerts = False
app.screen_updating = False

try:
    # --- Load the Add-in properly ---
    addin = app.books.open(addin_path)       # <<<<<< VERY IMPORTANT

    wb = app.books.open(crm_file)

    # Full recalc
    app.api.CalculateFull()

    # Save results back into Excel
    wb.save()

finally:
    wb.close()
    addin.close()     # optional
    app.quit()

print("Excel recalculated and saved successfully.")

# Step 2 — Pandas reads final values
crm_sheets_data = pd.read_excel(
    crm_file,
    sheet_name=None,
    engine='openpyxl'
)

crm_email_data = crm_sheets_data.get('Email', pd.DataFrame())

# Step 3 — Format numeric values only
def format_value(value):

    if pd.isna(value) or value is None:
        return value

    if isinstance(value, str):
        return value

    if isinstance(value, (int, float)):
        if value < 1 and value != 0:
            return f"{value:.2%}"
        else:
            return int(value)

    return value

crm_email_data['Value'] = crm_email_data['Value'].apply(format_value)

crm_email_data


In [ ]:
import pandas as pd
from datetime import datetime

def clean_and_prepare_data(crm_email_data):
    """
    Clean the data by removing whitespace and converting to lowercase for consistent matching
    """
    # Remove rows where Text is NaN or empty
    crm_email_data = crm_email_data.dropna(subset=['Text']).copy()
    
    # Clean the Text column: strip whitespace and convert to lowercase
    crm_email_data['Text'] = crm_email_data['Text'].str.strip().str.lower()
    
    # Clean the Value column: strip whitespace if it's a string
    crm_email_data['Value'] = crm_email_data['Value'].apply(
        lambda x: x.strip() if isinstance(x, str) else x
    )
    
    # Format numeric values
    def format_value(value):
        if pd.isna(value) or value == '':
            return value
        if isinstance(value, (int, float)):
            if value < 1 and value != 0:
                return f"{value:.2%}"
            else:
                return int(value)
        return value
    
    crm_email_data['Value'] = crm_email_data['Value'].apply(format_value)
    
    return crm_email_data

def generate_email_from_data(crm_email_data, output_file=None):
    """
    Generate email content from CRM email data and optionally save to file
    """
    
    # Clean the data first
    crm_email_data_clean = clean_and_prepare_data(crm_email_data)
    
    # Create a dictionary for easy access to values (with lowercase keys)
    data_dict = dict(zip(crm_email_data_clean['Text'], crm_email_data_clean['Value']))
    
    # Get current date for the email
    current_date = datetime.now().strftime("%d-%m-%Y")
    report_date = "20th November 2025"  # You can modify this as needed
    
    # Safe get function for missing keys
    def get_value(key, default="N/A"):
        return data_dict.get(key.strip().lower(), default)
    
    # Build the email content
    email_content = f"""Hello,

Below is the CRM user, activity and leads report for {report_date}

LEADS SUMMARY
{get_value('lead')} leads were generated in the system across all CS branches.
{get_value('percentage_accepted_lead')} ({get_value('accepted_lead')}) of leads generated were consented, {get_value('percentage_not_provided_lead')} ({get_value('not_provided_lead')}) were not provided and {get_value('percentage_rejected_lead')} ({get_value('rejected_lead')}) were rejected.
Out of {get_value('lead')} leads, {get_value('prospect_lead')} is a prospect.

image.png


MARKETING ACTIVITIES SUMMARY

Sales Agents

Total count of agents in CRM stood at {get_value('total_agent')}, and only {get_value('total_agent_logged_in')} logged in for the day.
Out of {get_value('agent_assigned_activities')} agents assigned activities for the day, {get_value('agent_completed_at_location')} ({get_value('percentage_agent_completed_at_location')}) agents completed at least one activity at the assigned location.
{get_value('agent_location_planned')} locations were planned for the day. Only {get_value('agent_reached_location')} ({get_value('percentage_reached_location')}) locations were reached on the day.
{get_value('agent_count_without_planned_location')} branches had no planned location visited by an agent. ({get_value('agent_branch_without_planned_location')})
{get_value('branches_count_without_assgned_activities')} branches had no assigned activities or planned locations to be visited ({get_value('branches_without_assgned_activities')}).

For today {current_date}
{get_value('todays_locations_planned')} locations have been planned.
{get_value('todays_agents_assigned')} ({get_value('percentage_todays_agents_assigned')}) have been assigned activities.
Average locations to be visited per agent is {get_value('average_location_agent_visited')}.

image.png


Team Leaders
Total count of TLs in CRM stood at {get_value('count_team_leaders')}, and only {get_value('logged_in_team_leaders')} TLs logged in for the day.
Out of {get_value('team_leaders_assigned_activities')} TLs assigned activities for the day, {get_value('team_leaders_completed_at_location')} ({get_value('percentage_completed_at_location')}) TLs completed at least one activity at the assigned location.
{get_value('team_leaders_location_planned')} locations were planned for the day. Only {get_value('team_leaders_location_reached')} ({get_value('percentage_tl_location_reached')}) locations were reached on the day.
{get_value('branches_tl_count_no_planned_location')} branches had no planned location visited by a TL. ({get_value('branches_tl_no_planned_location')})
{get_value('branches_tl_count_no_assigned_activites')} branches had not assigned any activities or locations to their TLs. ({get_value('branches_tl_no_assigned_activities')})

For today {current_date}
{get_value('todays_tls_location_planned')} locations have been planned.
{get_value('todays_tls_assigned_activities')} (90%) TLs have been assigned activities.
Average locations to be visited per TL is {get_value('average_location_visited_by_tl')}.

image.png


Best regards,
Raphael.
"""
    
    # Print the email content
    print(email_content)
    
    # Save to file if output file is specified
    if output_file:
        with open(output_file, 'w', encoding='utf-8') as file:
            file.write(email_content)
        print(f"\nEmail saved to: {output_file}")
    
    return email_content

def debug_data_keys(crm_email_data):
    """
    Helper function to debug and see all available keys after cleaning
    """
    cleaned_data = clean_and_prepare_data(crm_email_data)
    print("Available keys in cleaned data:")
    for key in cleaned_data['Text'].values:
        print(f"  '{key}'")
    print(f"\nTotal keys: {len(cleaned_data)}")
    return cleaned_data

# Main execution
if __name__ == "__main__":
    # Load your data
    crm_file = fr"C:\Users\Daniel\Desktop\code\CRM\CS\NEW_EXCEL\CS_CRM_24_11_2025.xlsx"
    crm_sheets_data = pd.read_excel(crm_file, sheet_name=None, engine='openpyxl')
    crm_email_data = crm_sheets_data.get('Email', pd.DataFrame())
    
    # Debug: Print available keys (optional - uncomment if needed)
    # print("Debugging data keys:")
    # cleaned_data = debug_data_keys(crm_email_data)
    
    # Generate and save the email
    output_filename = fr"C:\Users\Daniel\Desktop\code\CRM\CS\NEW_EXCEL\email_report_{datetime.now().strftime('%Y%m%d')}.txt"
    email_text = generate_email_from_data(crm_email_data, output_filename)

In [ ]:
Doris Lyakurwa <dorice@platinumcredit.co.tz>,
Sigfrid Mtei <sigfrid@platinumcredit.co.tz>,
Oscar Murigi <murigi@platinumcredit.co.ke>,
wayne@platinumcredit.co.ke,
Yusuph Khalid <yusuph@platinumcredit.co.tz>,
Allan Ruhuza <allan@platinumcredit.co.tz>,
Fragrance Mariki <fragrance@platinumcredit.co.tz>,
Vivian Karatta <vivian@platinumcredit.co.tz>,
Thomas Francis <thomas@platinumcredit.co.tz>,
Wilhelm Stephen <wilhelm@platinumcredit.co.tz>,
Regionalsalemanager@platinumcredit.co.tz,
Abraham Mlogho <abraham.mlogho.platinum@gmail.com>,
agostopher Mang'ati <agostopher.mangati.platinum@gmail.com>,
Mohamedi Omary <mohamedi.omar.platinum@gmail.com>,
Kelvin Mwasala <kelvin.mwasala@platinumcredit.co.tz>,
Daniel Masubi <daniel@platinumcredit.co.tz>,
Damson Daudi <damson@platinumcredit.co.tz>,
Felix Njeve <felix.njeve.platinum@gmail.com>,
Gerson Janga <gerson@platinumcredit.co.tz>

In [ ]:
dorice@platinumcredit.co.tz,
sigfrid@platinumcredit.co.tz,
murigi@platinumcredit.co.ke,
wayne@platinumcredit.co.ke,
yusuph@platinumcredit.co.tz,
allan@platinumcredit.co.tz,
fragrance@platinumcredit.co.tz,
vivian@platinumcredit.co.tz,
thomas@platinumcredit.co.tz,
wilhelm@platinumcredit.co.tz,
Regionalsalemanager@platinumcredit.co.tz,
abraham.mlogho.platinum@gmail.com,
agostopher.mangati.platinum@gmail.com,
mohamedi.omar.platinum@gmail.com,
kelvin.mwasala@platinumcredit.co.tz,
daniel@platinumcredit.co.tz,
damson@platinumcredit.co.tz,
felix.njeve.platinum@gmail.com,
gerson@platinumcredit.co.tz


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import os
from openpyxl import load_workbook
import numpy as np

def calculate_dynamic_column_widths(data):
    """
    Calculate column widths based on actual content length with better scaling
    """
    if not data or len(data) == 0:
        return [0.15]  # Default minimum width
    
    num_cols = len(data[0])
    max_content_lengths = [0] * num_cols
    
    # Find the maximum content length for each column
    for row in data:
        for col_idx, cell in enumerate(row):
            if col_idx < len(row):  # Safety check
                content_length = len(str(cell))
                max_content_lengths[col_idx] = max(max_content_lengths[col_idx], content_length)
    
    # Convert content lengths to column widths
    col_widths = []
    for max_len in max_content_lengths:
        # More generous width calculation
        # Base width + additional space for longer content
        base_width = 0.08  # Minimum width
        length_factor = max_len * 0.015  # More space per character
        width = min(base_width + length_factor, 0.4)  # Cap at maximum reasonable width
        col_widths.append(width)
    
    print(f"   Column widths calculated: {[f'{w:.3f}' for w in col_widths]}")
    return col_widths

def create_optimized_table_image(data, output_dir, filename, table_title=""):
    """
    Create table image with optimized column widths and better layout
    """
    try:
        if not data or len(data) == 0:
            print(f"   ❌ No data to create table for {filename}")
            return None
        
        # Calculate dynamic column widths
        col_widths = calculate_dynamic_column_widths(data)
        
        # Calculate figure size based on data and column widths
        total_width = sum(col_widths) * 120  # Scale factor for figure width
        num_rows = len(data)
        fig_width = max(16, min(total_width, 30))  # Reasonable min/max width
        fig_height = max(8, num_rows * 0.4)  # Dynamic height based on rows
        
        print(f"   Creating figure: {fig_width:.1f} x {fig_height:.1f} inches")
        
        fig, ax = plt.subplots(figsize=(fig_width, fig_height))
        ax.axis('off')
        
        # Add title if provided
        if table_title:
            plt.title(table_title, fontsize=14, fontweight='bold', pad=20)
        
        # Create table with optimized column widths
        table = ax.table(cellText=data,
                        cellLoc='center',
                        loc='center',
                        colWidths=col_widths)
        
        # Enhanced table styling
        table.auto_set_font_size(False)
        
        # Dynamic font sizing based on content length
        for key, cell in table.get_celld().items():
            row, col = key
            if row < len(data) and col < len(data[0]):
                cell_text = str(data[row][col])
                # Smaller font for longer content
                if len(cell_text) > 25:
                    cell.set_fontsize(20)
                elif len(cell_text) > 20:
                    cell.set_fontsize(20)
                elif len(cell_text) > 15:
                    cell.set_fontsize(20)
                else:
                    cell.set_fontsize(20)
        
        # Enhanced header styling
        if len(data) > 0:
            for i in range(len(data[0])):
                table[(0, i)].set_facecolor('#2E75B6')  # Header blue
                table[(0, i)].set_text_props(weight='bold', color='white')
                table[(0, i)].set_height(0.12)  # Header row height
        
        # Data row styling with better colors
        for i in range(1, len(data)):
            # Use softer alternating colors
            color = '#F0F8FF' if i % 2 == 0 else '#FFFFFF'  # Alice blue / White
            for j in range(len(data[0])):
                table[(i, j)].set_facecolor(color)
        
        # Enhanced cell borders
        for key, cell in table.get_celld().items():
            cell.set_edgecolor('#666666')  # Darker gray for better visibility
            cell.set_linewidth(0.8)
            cell.set_height(0.10)  # Consistent data row height
        
        # Better layout management
        plt.tight_layout(pad=3.0)  # More padding around the table
        
        # Save high quality image
        image_path = os.path.join(output_dir, filename)
        plt.savefig(image_path, dpi=200, bbox_inches='tight', pad_inches=0.5,
                   facecolor='white', edgecolor='none', 
                   transparent=False)
        plt.close()
        
        # Verify the image was created
        if os.path.exists(image_path) and os.path.getsize(image_path) > 1000:  # At least 1KB
            file_size_kb = os.path.getsize(image_path) / 1024
            print(f"   ✅ Image created successfully: {file_size_kb:.1f} KB")
            return image_path
        else:
            print(f"   ❌ Image file is too small or not created properly")
            return None
            
    except Exception as e:
        print(f"   ❌ Error creating optimized table image: {e}")
        import traceback
        traceback.print_exc()
        return None

def extract_and_clean_data(workbook, sheet_name, min_col, max_col, min_row, max_row):
    """
    Extract data with better cleaning and formatting
    """
    try:
        if sheet_name not in workbook.sheetnames:
            return None
        
        sheet = workbook[sheet_name]
        data = []
        
        print(f"   Extracting: {sheet_name} columns {min_col}-{max_col}, rows {min_row}-{max_row}")
        
        for row_idx, row in enumerate(sheet.iter_rows(min_row=min_row, max_row=max_row,
                                                     min_col=min_col, max_col=max_col,
                                                     values_only=True), start=min_row):
            # More aggressive cleaning - remove empty rows
            formatted_row = [format_cell_value(cell) for cell in row]
            
            # Check if row has substantial content (not just empty strings)
            has_content = any(str(cell).strip() not in ['', 'None', 'NaN'] for cell in formatted_row)
            
            if has_content:
                data.append(formatted_row)
                # Show sample of first few rows
                if len(data) <= 3:  # Only show first 3 rows for brevity
                    print(f"     Row {row_idx}: {[str(x)[:30] + '...' if len(str(x)) > 30 else x for x in formatted_row]}")
        
        if not data:
            print(f"   ⚠️ No substantial data found")
            return None
        
        print(f"   ✅ Extracted {len(data)} rows, {len(data[0])} columns")
        return data
        
    except Exception as e:
        print(f"   ❌ Extraction error: {e}")
        return None

def format_cell_value(value):
    """
    Format cell values: convert numbers < 1 to percentages, handle other types
    """
    if value is None or value == "" or str(value).strip() == "":
        return ""
    
    # Handle string cleanup first
    if isinstance(value, str):
        value = value.strip()
    
    try:
        # Try to convert to float
        float_val = float(value)
        
        # If value is between 0 and 1 (exclusive), format as percentage
        if 0 < float_val < 1:
            return f"{float_val:.2%}"
        # If value is exactly 1 or 0, format as percentage
        elif float_val == 1:
            return "100.00%"
        elif float_val == 0:
            return "0.00%"
        else:
            # For integers, remove decimal places if whole number
            if float_val.is_integer():
                return str(int(float_val))
            else:
                # For other numbers, keep 2 decimal places
                return f"{float_val:.2f}"
    except (ValueError, TypeError):
        # If not a number, return as string
        return str(value)

def create_wide_screenshots(excel_file_path, output_dir=None):
    """
    Main function to create screenshots with wide columns for better text display
    """
    if output_dir is None:
        output_dir = os.path.join(os.path.dirname(excel_file_path), "wide_screenshots")
    
    os.makedirs(output_dir, exist_ok=True)
    
    print("🔄 Creating wide-format screenshots...")
    
    try:
        workbook = load_workbook(excel_file_path, data_only=True)
        image_paths = {}
        
        # Configuration optimized for wider columns
        screenshots_config = [
            {
                'name': 'leads_summary',
                'sheet': 'LEADS_SUMMARY',
                'range': (2, 7, 1, 15),
                'filename': 'leads_summary_wide.png',
                'title': 'LEADS SUMMARY'
            },
            {
                'name': 'agent_summary', 
                'sheet': 'summary',
                'range': (2, 16, 1, 25),  # B:P
                'filename': 'agent_summary_wide.png',
                'title': 'AGENT SUMMARY'
            },
            {
                'name': 'team_leader_summary',
                'sheet': 'summary', 
                'range': (19, 32, 1, 25),  # S:AF
                'filename': 'team_leader_summary_wide.png',
                'title': 'TEAM LEADER SUMMARY'
            }
        ]
        
        for config in screenshots_config:
            print(f"\n🎯 Processing {config['name']}...")
            
            min_col, max_col, min_row, max_row = config['range']
            data = extract_and_clean_data(workbook, config['sheet'], 
                                        min_col, max_col, min_row, max_row)
            
            if data and len(data) > 0:
                # Create the optimized table image
                image_path = create_optimized_table_image(
                    data, output_dir, config['filename'], config.get('title', '')
                )
                
                if image_path:
                    image_paths[config['name']] = image_path
                    print(f"  ✅ Wide-format {config['name']} created successfully!")
                else:
                    print(f"  ❌ Failed to create wide-format image for {config['name']}")
            else:
                print(f"  ❌ No data extracted for {config['name']}")
        
        workbook.close()
        
        print(f"\n📊 Final Results:")
        for name, path in image_paths.items():
            if os.path.exists(path):
                file_size_kb = os.path.getsize(path) / 1024
                print(f"   📸 {name}: {path} ({file_size_kb:.1f} KB)")
            else:
                print(f"   ❌ {name}: FILE NOT FOUND!")
        
        return image_paths
        
    except Exception as e:
        print(f"❌ Error in wide screenshot process: {e}")
        import traceback
        traceback.print_exc()
        return {}

# Main execution
if __name__ == "__main__":
    crm_file = fr"C:\Users\Daniel\Desktop\code\CRM\CS\NEW_EXCEL\CS_CRM_24_11_2025.xlsx"
    
    print("🚀 Starting wide-format screenshot creation...")
    screenshot_paths = create_wide_screenshots(crm_file)
    
    if screenshot_paths:
        print(f"\n🎉 Successfully created {len(screenshot_paths)} wide-format screenshots!")
        print("📧 These should now display all text properly in your email!")
    else:
        print("\n💥 Failed to create wide-format screenshots!")